In [ ]:
import os
import re
import json
import datetime
import time
import random
import glob

# 路径与配置
DATA_DIR = "./data"
SRC_MD = f"{DATA_DIR}/blackwukong.md"
TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_BASE_JSONL = f"{DATA_DIR}/wukong_base_{TS}.jsonl"
OUT_JSONL = f"{DATA_DIR}/wukong_dataset_{TS}.jsonl"

# from openai import OpenAI

# # 注意：为演示方便，这里直接在代码中写入密钥与模型，不推荐在生产环境硬编码敏感信息，建议改用环境变量或密钥管理服务
# BASE_URL = "https://api.siliconflow.cn/v1"
# MODEL_ID = "Qwen/Qwen3-235B-A22B-Instruct-2507"
# API_KEY = "sk-xxx"

# client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
# print(f"Using model: {MODEL_ID} @ {BASE_URL}")

In [ ]:
with open(SRC_MD, "r", encoding="utf-8") as f:
    raw_markdown = f.read()

md_src = raw_markdown

# 按标题切分；无标题回退按段落
matches = list(re.finditer(r"(?m)^(#{2,3})\s+(.+)$", md_src))
sections = []
if not matches:
    paras = [p.strip() for p in re.split(r"\n\s*\n", md_src) if len(p.strip()) >= 100]
    sections = paras
else:
    for i, m in enumerate(matches):
        s = m.start()
        e = matches[i + 1].start() if i + 1 < len(matches) else len(md_src)
        block = md_src[s:e].strip()
        if len(block) >= 100:
            sections.append(block)

# 去重保序
seen = set()
uniq = []
for t in sections:
    key = re.sub(r"\s+", " ", t).lower()[:240]
    if key in seen:
        continue
    seen.add(key)
    uniq.append(t)
sections = uniq

print(f"sections={len(sections)}")